# 05 Positional Encodings & RoPE (PyTorch Modules & Frequency Plots)

Implementing Sinusoidal Positional Encoding and Rotary Position Embeddings (RoPE) in PyTorch, and plotting vector rotations and NTK frequency scaling.

## Part 1: Sinusoidal Positional Encoding Heatmap & Waves
Plotting the 2D PE matrix and frequency waves.

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

plt.style.use('dark_background')

class PyTorchSinusoidalPE(nn.Module):
    def __init__(self, d_model=128, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

pe_module = PyTorchSinusoidalPE()
pe_tensor = pe_module.pe.numpy()

plt.figure(figsize=(10, 4))
plt.imshow(pe_tensor, cmap='magma', aspect='auto')
plt.title('PyTorch Sinusoidal Positional Encoding Matrix (100 pos x 128 dims)')
plt.xlabel('Embedding Dimension'); plt.ylabel('Sequence Position')
plt.colorbar(); plt.show()

## Part 2: Rotary Position Embeddings (RoPE) PyTorch Module
Implementing 2D vector rotation projections.

In [ ]:
class PyTorchRoPE(nn.Module):
    def __init__(self, dim=64, theta=10000.0):
        super().__init__()
        self.dim = dim
        inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer('inv_freq', inv_freq)
    def forward(self, seq_len):
        t = torch.arange(seq_len, dtype=torch.float32)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()

rope = PyTorchRoPE(dim=64)
cos_c, sin_c = rope(seq_len=30)
print('PyTorch RoPE Cosine Tensor Cache Shape:', cos_c.shape)

## Part 3: PyTorch RoPE Relative Score Distance Decay
Evaluating rotated inner products $\langle R_m q, R_n k \rangle$ over token distance offsets $|m-n| \in [0, 30]$.

In [ ]:
q_dummy = torch.ones(1, 64) / math.sqrt(64)
k_dummy = torch.ones(1, 64) / math.sqrt(64)
dot_products = []
for delta in range(30):
    cos_d, sin_d = cos_c[delta:delta+1], sin_c[delta:delta+1]
    k1, k2 = k_dummy[:, :32], k_dummy[:, 32:]
    k_rot = torch.cat((-k2, k1), dim=-1)
    k_transformed = (k_dummy * cos_d) + (k_rot * sin_d)
    dot_products.append(torch.sum(q_dummy * k_transformed).item())

plt.figure(figsize=(8, 4))
plt.plot(range(30), dot_products, marker='o', color='#38d9a9', lw=2)
plt.title('PyTorch RoPE Relative Score vs Token Distance (|m-n|)')
plt.xlabel('Relative Distance (|m-n|)'); plt.ylabel('Rotated Inner Product Score')
plt.grid(True); plt.show()

## Part 4: Context Length Extension Frequency Spectrum (Original vs Linear vs NTK-Aware)
Comparing rotation frequency curves $\theta_i$ across dimensions.

In [ ]:
d_head = 128
i_dims = torch.arange(0, d_head // 2, dtype=torch.float32)
base_orig, s = 10000.0, 4.0

freq_orig = 1.0 / (base_orig ** (2 * i_dims / d_head))
freq_linear = freq_orig / s
base_ntk = base_orig * (s ** (d_head / (d_head - 2)))
freq_ntk = 1.0 / (base_ntk ** (2 * i_dims / d_head))

plt.figure(figsize=(9, 4.5))
plt.plot(i_dims.numpy(), freq_orig.numpy(), label='Original (4k Context)', color='#4dabf7', lw=2.5)
plt.plot(i_dims.numpy(), freq_linear.numpy(), label='Linear Scaling', color='#ff6b6b', lw=2, linestyle='--')
plt.plot(i_dims.numpy(), freq_ntk.numpy(), label='NTK-Aware Base Scaling', color='#51cf66', lw=2.5)
plt.yscale('log')
plt.title('RoPE Context Extension Frequency Spectrum across Vector Dimensions')
plt.xlabel('Dimension Index Pair (i)'); plt.ylabel('Rotation Frequency (log scale)')
plt.grid(True, which='both'); plt.legend(); plt.show()